# FloorPlanTo3D — Colab Smoke Test

This notebook validates the complete training pipeline on a small subset of the CubiCasa5K dataset before booking time on the A100 supercomputer.

## What this notebook does

1. **Verifies the GPU** — Colab gives you a T4 or similar by default. Confirms torch sees it.
2. **Mounts Google Drive** — so the dataset lives persistently across Colab sessions.
3. **Installs the missing dependencies** — only what Colab doesn't already pre-install.
4. **Downloads CubiCasa5K once** — about 5.5 GB. Stored in your Drive so subsequent runs skip this step.
5. **Extracts the dataset once** — about 10-12 GB extracted. Stored in your Drive.
6. **Runs the SVG → COCO converter** — produces train/val annotation files.
7. **Inspects the conversion report** — shows per-class annotation counts; flags any class with too few annotations to train.
8. **Smoke-tests training** — 50 plans, 2 epochs, batch size 2 — about 15 minutes on a T4.
9. **Checks the per-class IoU CSV** — confirms the IoU monitor is producing data.

## What this notebook does NOT do

- Train a usable model. The smoke test only validates the pipeline runs end-to-end. The real training run happens on the supercomputer with the full dataset and 50+ epochs.
- Use the production Dockerfile. Colab is a managed Python environment, not a container runtime.

## When to abort

If any cell produces an error, **stop and report it** before continuing. Each cell is independent enough that subsequent cells will likely fail confusingly if an earlier one didn't work.

## Estimated time

- First run (with download + extract): **~45 minutes** (dataset download dominates)
- Subsequent runs (dataset already in Drive): **~25 minutes** (just smoke-test training)

## Section 0 — Upload the project zip

Before running anything else, you need the project code in Colab.

**Two ways to do this:**

**Option A (recommended):** Upload `FloorPlan_Fixed.zip` to your Google Drive at `/MyDrive/floorplan/FloorPlan_Fixed.zip`. The notebook will read it from there.

**Option B:** Run the cell below to manually upload via the file picker (only works in Colab, and the zip is re-uploaded every session — slower).

In [ ]:
# -- Paths you may need to adjust -----------------------------------------
DRIVE_PROJECT_ZIP = "/content/drive/MyDrive/floorplan/FloorPlan_Fixed.zip"
DRIVE_DATA_DIR    = "/content/drive/MyDrive/floorplan/data"
WORK_DIR          = "/content/work"    # ephemeral local SSD -- fast random I/O

# -- What lives in Drive (persistent across sessions) ---------------------
#   cubicasa5k.zip   -> DRIVE_DATA_DIR/cubicasa5k.zip   (downloaded once)
#   training results -> DRIVE_RESULTS_DIR               (saved at the end)
CUBICASA_ZIP      = f"{DRIVE_DATA_DIR}/cubicasa5k.zip"
DRIVE_RESULTS_DIR = f"{DRIVE_DATA_DIR}/training_results_smoke"

# -- What lives on local disk (re-created each session) -------------------
#   CUBICASA_ROOT:    unzipped from Drive zip each session  (~2 min, local SSD)
#   COCO_OUTPUT_DIR:  generated by converter                (~1 min)
#   TRAIN_OUTPUT_DIR: checkpoints + IoU CSV; copied to Drive at the end
#
# WHY NOT DRIVE for these? Drive has ~5-10ms latency per file open.
# The DataLoader opens thousands of PNGs per epoch -- on Drive that would
# make each epoch 5-10x slower than reading from local SSD.
PROJECT_DIR       = f"{WORK_DIR}/Floor_Fixed"
CUBICASA_ROOT     = f"{WORK_DIR}/cubicasa5k"
COCO_OUTPUT_DIR   = f"{WORK_DIR}/cubicasa_coco_smoke50"
TRAIN_OUTPUT_DIR  = f"{WORK_DIR}/training_smoke_test"

print("Configuration:")
print(f"  Project zip (Drive):  {DRIVE_PROJECT_ZIP}")
print(f"  CubiCasa zip (Drive): {CUBICASA_ZIP}")
print(f"  Results -> Drive:     {DRIVE_RESULTS_DIR}")
print()
print(f"  Work dir (local SSD): {WORK_DIR}")
print(f"  CubiCasa root:        {CUBICASA_ROOT}")
print(f"  COCO output:          {COCO_OUTPUT_DIR}")
print(f"  Training output:      {TRAIN_OUTPUT_DIR}")

## Section 1 — Verify GPU availability

Before doing any work, confirm Colab has given us a GPU. If this cell shows "No GPU available," go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or A100 if you have Pro+), then re-run from the top.

The free tier gives a T4. That's enough for the smoke test. The real training run will happen on the supercomputer.

In [ ]:
import torch, platform, sys

print(f"Python:         {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:   {torch.version.cuda}")
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU memory:     {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
else:
    print()
    print("❌ NO GPU AVAILABLE")
    print("   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")
    print("   Then re-run this notebook from the top.")
    raise SystemExit("No GPU. Cannot continue.")

print()
print("✅ GPU ready")

## Section 2 — Mount Google Drive

Drive is where we persist the dataset, the converted COCO files, and (optionally) intermediate checkpoints. Without Drive, every Colab session would re-download the 5.5 GB CubiCasa zip.

When you run this cell, Colab will prompt you to authorize Drive access. Click through the prompts and select the Google account that has the dataset zip.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

# Verify it actually mounted
if not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError("Drive mount failed — re-run this cell")

# Create the directories we'll use
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

print(f"✅ Drive mounted")
print(f"   Data dir: {DRIVE_DATA_DIR}")
print(f"   Work dir: {WORK_DIR}")

## Section 3 — Unpack the project code

Extracts `FloorPlan_Fixed.zip` from Drive into `/content/work/Floor_Fixed`. This is recreated every session because Colab's `/content` is ephemeral, but the extraction is fast (~5 seconds).

In [ ]:
import os, zipfile, shutil

if not os.path.exists(DRIVE_PROJECT_ZIP):
    raise FileNotFoundError(
        f"Project zip not found at {DRIVE_PROJECT_ZIP}\n"
        f"Upload FloorPlan_Fixed.zip to /MyDrive/floorplan/ in Google Drive, then re-run this cell."
    )

# Clean the work directory if it exists from a previous session
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Extract
with zipfile.ZipFile(DRIVE_PROJECT_ZIP) as zf:
    zf.extractall(WORK_DIR)

# Sanity check: confirm the expected directory structure exists
required = [
    f"{PROJECT_DIR}/train_mask2former.py",
    f"{PROJECT_DIR}/config/classes.py",
    f"{PROJECT_DIR}/scripts/convert_cubicasa_to_coco.py",
    f"{PROJECT_DIR}/requirements.txt",
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"Project extraction incomplete. Missing: {missing}")

print(f"✅ Project extracted to {PROJECT_DIR}")
print(f"   Files: {sum(len(fs) for _, _, fs in os.walk(PROJECT_DIR))} total")

## Section 4 — Install dependencies

Colab pre-installs `transformers`, `accelerate`, and `peft` but often at
versions that conflict with each other after updates. This cell uninstalls
all three and reinstalls the exact version matrix known to work together:

| Package | Version |
|---|---|
| transformers | 4.41.0 |
| accelerate | 0.30.1 |
| peft | 0.11.1 |
| torchmetrics | 1.3.2 |

`peft==0.11.1` is the patch that fixes the `clear_device_cache` ImportError.
`accelerate==0.30.1` is the newest version compatible with both.

**After this cell completes, continue directly to Section 5.** No runtime
restart is needed because we did a clean uninstall first.

In [ ]:
# Uninstall whatever Colab pre-installed to get a clean slate,
# then install the exact version matrix that is known to work together.
# (peft==0.11.1 patches the clear_device_cache ImportError; accelerate==0.30.1
#  is the newest version compatible with both transformers 4.41.0 and peft 0.11.1)
!pip uninstall -y transformers accelerate peft torchmetrics
!pip install -q transformers==4.41.0 accelerate==0.30.1 peft==0.11.1 torchmetrics==1.3.2

import transformers, accelerate, peft, torchmetrics
print(f"transformers:  {transformers.__version__}")
print(f"accelerate:    {accelerate.__version__}")
print(f"peft:          {peft.__version__}")
print(f"torchmetrics:  {torchmetrics.__version__}")
print()
print(" Dependencies OK")

## Section 5 — Download CubiCasa5K (once)

CubiCasa5K is hosted on Zenodo. The zip is about 5.5 GB.

**This step skips itself if the zip already exists in Drive.** First-time run takes ~10-15 minutes depending on Colab's network speed. Subsequent runs take a few seconds (just the existence check).

If the download fails mid-way and leaves a partial file, delete the partial file from Drive and re-run.

In [ ]:
import os, subprocess

CUBICASA_URL = "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
EXPECTED_SIZE_GB = 5.0   # rough lower bound — flag if download is suspiciously small

if os.path.exists(CUBICASA_ZIP):
    size_gb = os.path.getsize(CUBICASA_ZIP) / 1e9
    if size_gb < EXPECTED_SIZE_GB:
        print(f"⚠️ Existing zip is only {size_gb:.2f} GB — looks incomplete.")
        print(f"   Deleting and re-downloading.")
        os.remove(CUBICASA_ZIP)
    else:
        print(f"✅ CubiCasa zip already in Drive ({size_gb:.2f} GB) — skipping download")

if not os.path.exists(CUBICASA_ZIP):
    print(f"Downloading from {CUBICASA_URL}")
    print(f"Target: {CUBICASA_ZIP}")
    print(f"This is ~5.5 GB and takes 10-15 minutes...")
    print()
    # Use wget with progress and continue-on-error
    result = subprocess.run(
        ["wget", "--continue", "--progress=dot:giga",
         "-O", CUBICASA_ZIP, CUBICASA_URL],
        check=False
    )
    if result.returncode != 0:
        # Clean up partial download
        if os.path.exists(CUBICASA_ZIP):
            partial_gb = os.path.getsize(CUBICASA_ZIP) / 1e9
            print(f"⚠️ Download failed at {partial_gb:.2f} GB")
        raise RuntimeError(f"wget exited with code {result.returncode}")

    final_gb = os.path.getsize(CUBICASA_ZIP) / 1e9
    print(f"\n✅ Downloaded {final_gb:.2f} GB to {CUBICASA_ZIP}")

## Section 6 — Extract CubiCasa5K to local disk

The zip lives in Drive (persistent). We extract it to the **local SSD**
(`WORK_DIR`) every session. Extraction takes about 2 minutes, but every
subsequent operation (converter, DataLoader) then reads from fast local
I/O instead of Drive.

**Why re-extract every session?** Colab's `/content` is ephemeral and
disappears when the session ends. The Drive zip is the durable source;
the local tree is a fast working copy. Two minutes of extraction is a
good trade against 5-10x slower training caused by Drive per-file latency.

In [ ]:
import os, zipfile, time, shutil

# Always extract fresh to local WORK_DIR (ephemeral SSD).
# We never skip this step because WORK_DIR is gone after each session.
if os.path.exists(CUBICASA_ROOT):
    print(f"Removing stale extraction at {CUBICASA_ROOT} ...")
    shutil.rmtree(CUBICASA_ROOT)

os.makedirs(CUBICASA_ROOT, exist_ok=True)
print(f"Extracting {CUBICASA_ZIP}")
print(f"  source : Drive (network read)")
print(f"  target : {CUBICASA_ROOT} (local SSD write)")
print(f"  ETA    : ~2 minutes")
t0 = time.time()

with zipfile.ZipFile(CUBICASA_ZIP) as zf:
    members = zf.namelist()
    print(f"   {len(members)} files to extract")
    for i, m in enumerate(members):
        zf.extract(m, CUBICASA_ROOT)
        if (i + 1) % 500 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(members) - (i + 1)) / rate / 60
            print(f"   {i+1:>5d}/{len(members)} files  "
                  f"({rate:.0f} files/sec, ETA {eta:.1f} min)")

elapsed = time.time() - t0
print(f"\n Extracted in {elapsed/60:.1f} minutes")

# CubiCasa archives sometimes nest contents inside a 'cubicasa5k/' subfolder.
# Walk to find the actual root that has train.txt and val.txt.
def find_dataset_root(start):
    for root, dirs, files in os.walk(start):
        if "train.txt" in files and "val.txt" in files:
            return root
    return None

actual_root = find_dataset_root(CUBICASA_ROOT)
if actual_root is None:
    raise RuntimeError(
        f"Could not find train.txt / val.txt under {CUBICASA_ROOT}")
if actual_root != CUBICASA_ROOT:
    print(f"   Archive was nested. Updating CUBICASA_ROOT -> {actual_root}")
    CUBICASA_ROOT = actual_root

n_train = sum(1 for _ in open(f"{CUBICASA_ROOT}/train.txt"))
n_val   = sum(1 for _ in open(f"{CUBICASA_ROOT}/val.txt"))
print(f" Using CUBICASA_ROOT = {CUBICASA_ROOT}")
print(f"   train.txt: {n_train} plans  |  val.txt: {n_val} plans")
print(f" Local disk ready for fast I/O")

## Section 7 — Run the SVG → COCO converter

This is where your converter (`scripts/convert_cubicasa_to_coco.py`) does its work. For the smoke test, we cap at **50 plans per split** to keep things fast.

Expected runtime: **30-90 seconds** for 100 plans total. Output goes to `COCO_OUTPUT_DIR` in Drive so subsequent training runs can reuse it.

If you want to do a full-dataset conversion later (for the real training), re-run this cell with `--max-plans-per-split` removed.

In [ ]:
import os, subprocess, sys

# Wipe previous output for a clean smoke test (idempotent run)
if os.path.exists(COCO_OUTPUT_DIR):
    import shutil
    shutil.rmtree(COCO_OUTPUT_DIR)

env = os.environ.copy()
env["PYTHONPATH"] = PROJECT_DIR

result = subprocess.run(
    [sys.executable, f"{PROJECT_DIR}/scripts/convert_cubicasa_to_coco.py",
     "--cubicasa-root", CUBICASA_ROOT,
     "--output-dir",    COCO_OUTPUT_DIR,
     "--max-plans-per-split", "50",
     "--splits", "train", "val"],
    env=env, capture_output=True, text=True
)

# Print the converter's output regardless of exit code
print("=== CONVERTER STDOUT ===")
print(result.stdout)
if result.stderr:
    print("=== CONVERTER STDERR ===")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Converter exited with code {result.returncode}")

# Verify the output files exist
required_files = [
    f"{COCO_OUTPUT_DIR}/train/annotations.json",
    f"{COCO_OUTPUT_DIR}/val/annotations.json",
    f"{COCO_OUTPUT_DIR}/conversion_report.json",
]
missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"Converter completed but expected files are missing: {missing}")

print()
print(f"✅ Conversion complete. Output: {COCO_OUTPUT_DIR}")

## Section 8 — Inspect the conversion report

This is **the most important diagnostic step** in the whole notebook. Even on a 50-plan subset, the per-class counts tell us:

- **Which classes will train well** (high count = the model has enough examples to learn)
- **Which classes will train weakly** (low count = warning sign)
- **Which classes will not train at all** (zero count = the converter found no examples in the smoke subset)

For the smoke test, ZERO counts on some classes are expected and OK — the subset is small. The real question is whether the converter ran successfully and produced annotations for the major classes (wall, window, door, room types).

In [ ]:
import json

with open(f"{COCO_OUTPUT_DIR}/conversion_report.json") as f:
    report = json.load(f)

# Load class names from the project
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from config.classes import PROJECT_ID_TO_NAME, NUM_CLASSES

for split_name, st in report["splits"].items():
    print(f"=== {split_name} ===")
    print(f"  plans requested:  {st['plans_requested']}")
    print(f"  plans converted:  {st['plans_converted']}")
    print(f"  plans skipped:    {st['plans_skipped']}")
    if st['skipped_reasons']:
        print(f"  skip reasons:")
        for reason, count in sorted(st['skipped_reasons'].items(),
                                     key=lambda x: -x[1]):
            print(f"    {count:>6d}  {reason}")
    print(f"  annotations by class:")
    for pid in sorted(PROJECT_ID_TO_NAME):
        count = st['annotations_by_class'].get(str(pid), 0)
        if count == 0:
            count = st['annotations_by_class'].get(pid, 0)
        name  = PROJECT_ID_TO_NAME[pid]
        flag  = "  ⚠️  ZERO" if count == 0 else ("  ⚠️  FEW" if count < 10 else "")
        print(f"    pid={pid:>2d}  {name:<15s}  {count:>8d}{flag}")
    print()

# Decision check for the smoke test:
# We need at least walls to be present, otherwise something is very wrong.
train_stats = report["splits"]["train"]
wall_count = (train_stats["annotations_by_class"].get(1, 0)
              or train_stats["annotations_by_class"].get("1", 0))
if wall_count < 50:
    raise RuntimeError(
        f"Only {wall_count} wall annotations in 50-plan train subset — "
        f"something is wrong with conversion. Expected ~500-1500 walls."
    )
print(f"✅ Conversion sanity check: {wall_count} wall annotations in train set")

## Section 9 — Smoke-test training (2 epochs)

Now we run the actual training script for **just 2 epochs** on the 50-plan subset. This is the real validation: it exercises the data loader, the model forward/backward pass, the loss computation, the checkpoint save, and the per-class IoU callback (Item 2).

**Expected runtime: 10-15 minutes** on a free T4.

**What success looks like:**
- The training loss decreases between epochs (even if absolute value is high)
- No CUDA OOM errors (50 plans × batch 2 should fit easily)
- A `per_class_iou.csv` file appears in the output directory
- A checkpoint folder appears in the output directory

**What failure looks like:**
- `CUDA out of memory` → batch size needs to be smaller (try `--batch_size 1`)
- `KeyError` or `IndexError` in data loading → bug in the converter output that 50 plans didn't surface in the toy tests
- `Cannot find file` → COCO `file_name` references images that don't exist (path mismatch)

The training script's `Item 1` consistency check runs first and validates `config/classes.py` is internally consistent. If that fails, it'll tell you exactly which class mapping is wrong.

In [ ]:
import os, subprocess, sys, time

# Clean previous training output (idempotent run)
if os.path.exists(TRAIN_OUTPUT_DIR):
    import shutil
    shutil.rmtree(TRAIN_OUTPUT_DIR)
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = PROJECT_DIR

# Find a sane base model name. Mask2Former-Swin-Large is the production target.
# For a smoke test we use the smaller Swin-Tiny variant — same architecture
# semantics but downloads ~200 MB instead of ~800 MB and runs ~3x faster.
SMOKE_TEST_MODEL = "facebook/mask2former-swin-tiny-coco-instance"

cmd = [
    sys.executable, f"{PROJECT_DIR}/train_mask2former.py",
    "--dataset_dir", COCO_OUTPUT_DIR,
    "--output_dir",  TRAIN_OUTPUT_DIR,
    "--base_model",  SMOKE_TEST_MODEL,
    "--epochs", "2",
    "--batch_size", "2",
    "--iou_eval_images", "10",
    # eval_steps=3: with 50 plans / effective_batch 16 = ~3 steps/epoch,
    # this ensures evaluation fires after every epoch regardless of auto-cap.
    "--eval_steps", "3",
    "--save_steps", "3",
]
print(f"Running: {' '.join(cmd)}")
print(f"Expected runtime: 10-15 minutes")
print()

# Stream output line by line so we can watch progress in real time
t0 = time.time()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()

elapsed = (time.time() - t0) / 60
print()
print(f"Training exited after {elapsed:.1f} minutes with code {proc.returncode}")

if proc.returncode != 0:
    raise RuntimeError(f"Training script failed — see output above for the error.")

## Section 10 — Inspect the per-class IoU CSV

The per-class IoU callback we built in Item 2 writes one row per evaluation step, with one column per class. Even at 2 epochs of smoke test, this file should exist and have values — most of them small (random initialization) but non-NaN.

**What to look at:**
- The CSV exists at all
- It has the expected columns (one per class)
- Values are floats, not NaN
- IoU for the most common class (wall) should be > 0 after 2 epochs (even if small)

This is mostly a plumbing check. Real IoU evaluation happens after 30+ epochs of training on the full dataset.

In [ ]:
import os, pandas as pd

csv_path = f"{TRAIN_OUTPUT_DIR}/per_class_iou.csv"

if not os.path.exists(csv_path):
    print(f"❌ per_class_iou.csv not found at {csv_path}")
    print("   The PerClassIoUCallback may not have triggered, or the trainer didn't reach eval.")
    print("   Check the training output above for warnings.")
    # Don't raise — the smoke test mostly passed if training completed
else:
    df = pd.read_csv(csv_path)
    print(f"✅ Per-class IoU CSV: {csv_path}")
    print(f"   Rows: {len(df)} (one per eval step)")
    print(f"   Columns: {list(df.columns)}")
    print()
    print("Last evaluation row:")
    last = df.iloc[-1]
    for col, val in last.items():
        if col in ("step", "epoch"):
            print(f"  {col:<20s}  {val}")
        else:
            try:
                v = float(val)
                marker = " ⚠️ NaN" if pd.isna(v) else ""
                print(f"  {col:<20s}  IoU={v:.4f}{marker}")
            except (ValueError, TypeError):
                print(f"  {col:<20s}  {val}")

## Section 11 — Final summary

If you got this far without errors, the pipeline is ready for the supercomputer.

**What you've now validated:**

✅ Python/PyTorch environment can run the project
✅ Project zip extracts and is structurally complete
✅ Dependencies install cleanly
✅ CubiCasa dataset downloads and extracts
✅ The SVG → COCO converter runs and produces valid annotations
✅ The COCO output passes `train_mask2former.py`'s data loader
✅ The model trains for 2 epochs without crashing
✅ The per-class IoU callback (Item 2) writes its CSV

**Before booking supercomputer time, decide:**

1. **Class mapping** — review the conversion report's per-class counts on the FULL dataset (not just 50 plans). If any class has truly zero annotations, decide whether to drop it from `config/classes.py` or accept the weak performance.

2. **Run the full conversion** — re-run section 7 without `--max-plans-per-split` to produce the full-dataset COCO files. This takes ~20-30 minutes and writes to Drive.

3. **Pick the production model** — replace `facebook/mask2former-swin-tiny-coco-instance` with `facebook/mask2former-swin-large-coco-instance` for the real run. This downloads ~800 MB and uses ~4-5x more GPU memory but produces much better results.

4. **Tune batch size** — A100 80GB can handle batch size 8-16 at 1024×1024 with Swin-Large. A100 40GB realistically handles 4-8. Use `--grad_accum` to keep effective batch size high if your physical batch is small (effective batch = batch_size × grad_accum). Add `--fp16` for mixed-precision training, which cuts memory ~40% and speeds up A100 throughput.

5. **Plan epochs** — 50-80 epochs is reasonable for fine-tuning Mask2Former on 5000 plans. Check the per-class IoU CSV every few epochs. The training script also accepts `--lr`, `--backbone_lr_ratio`, `--warmup_ratio`, `--save_steps`, and `--eval_steps` for finer control.

### Example full-run command for the supercomputer

```
python train_mask2former.py \
    --dataset_dir   /path/to/cubicasa_coco_full \
    --output_dir    /path/to/training_run \
    --base_model    facebook/mask2former-swin-large-coco-instance \
    --epochs        80 \
    --batch_size    4 \
    --grad_accum    4 \
    --fp16 \
    --iou_eval_images 50
```

This gives an effective batch size of 16 (4 × 4) with mixed precision. Adjust `--batch_size` up if your A100 has headroom.

**Report any errors from this notebook back to the assistant so we can fix them before the real run.**

## Section 12 — Save training results to Drive

`TRAIN_OUTPUT_DIR` lives on the ephemeral local SSD and will be **permanently
deleted** when this Colab session ends. This cell copies everything in that
directory (checkpoints, `per_class_iou.csv`, trainer state) to a persistent
folder in your Google Drive before the session can time out.

**Run this cell immediately after training completes.** If the browser
closes or the session times out first, the results are gone.

What gets saved:
- `checkpoint-*/` -- model weights at each save step
- `per_class_iou.csv` -- per-class IoU after each eval step
- `trainer_state.json` -- HuggingFace Trainer metadata (step, loss history)
- `training_args.bin` -- the exact arguments the training run used

In [ ]:
import os, shutil, time

if not os.path.exists(TRAIN_OUTPUT_DIR):
    print('TRAIN_OUTPUT_DIR does not exist -- training may not have run yet.')
    print(f'Expected: {TRAIN_OUTPUT_DIR}')
else:
    # Timestamp the destination so multiple runs do not overwrite each other.
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    save_dest = f"{DRIVE_RESULTS_DIR}_{timestamp}"

    print(f'Copying training results to Drive ...')
    print(f'  source : {TRAIN_OUTPUT_DIR}')
    print(f'  dest   : {save_dest}')

    os.makedirs(os.path.dirname(save_dest), exist_ok=True)
    shutil.copytree(TRAIN_OUTPUT_DIR, save_dest)

    saved_files = []
    for root, dirs, files in os.walk(save_dest):
        for f in files:
            p = os.path.join(root, f)
            size_mb = os.path.getsize(p) / 1e6
            saved_files.append((p.replace(save_dest, ''), size_mb))

    total_mb = sum(s for _, s in saved_files)
    print(f'\nSaved {len(saved_files)} files ({total_mb:.1f} MB total):')
    for rel, size_mb in sorted(saved_files):
        print(f'  {size_mb:8.2f} MB  {rel}')

    iou_csv = os.path.join(save_dest, 'per_class_iou.csv')
    if os.path.exists(iou_csv):
        print(f'\nper_class_iou.csv saved at: {iou_csv}')
    else:
        print('\nNote: per_class_iou.csv not found -- OK if training ran fewer than eval_steps steps.')

    print(f'\nAll results safe in Drive at:')
    print(f'  {save_dest}')
    print('Session can now be closed safely.')